# 使用 Meta 系列模型構建

## 簡介

本課程將涵蓋：

- 探索兩個主要的 Meta 系列模型 - Llama 3.1 與 Llama 3.2
- 理解每個模型的使用案例與場景
- 程式碼範例展示各模型的獨特功能


## Meta 系列模型

在本課程中，我們將探討 Meta 系列或「Llama 族群」中的兩個模型 - Llama 3.1 與 Llama 3.2

這些模型有不同的變體，並可在 [Microsoft Foundry Models catalog](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) 中找到。

> **注意：** GitHub Models 將於 2026 年 7 月底退役。以下有更多使用 [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) 進行 AI 模型原型設計的詳細資訊。

模型變體：
- Llama 3.1 - 70B 指令型
- Llama 3.1 - 405B 指令型
- Llama 3.2 - 11B 視覺指令型
- Llama 3.2 - 90B 視覺指令型

*注意：Llama 3 也在 Microsoft Foundry Models 中提供，但本課程不包括該模型*



## Llama 3.1 

擁有 4050 億參數的 Llama 3.1 屬於開源大型語言模型（LLM）範疇。 

此版本是對早期發佈的 Llama 3 的升級，提供： 

- 更大的上下文視窗 - 128k 代幣 對比 8k 代幣 
- 更大的最大輸出代幣數 - 4096 對比 2048 
- 更佳的多語言支持 - 由於訓練代幣數量增加 

這些使得 Llama 3.1 能夠在建構生成式 AI 應用時處理更複雜的使用案例，包括： 
- 原生函數調用 - 能夠調用 LLM 工作流程以外的外部工具與函數
- 更佳的 RAG 表現 - 由於更大的上下文視窗 
- 合成數據生成 - 能夠為微調等任務創建有效數據 



### 原生函數調用 

Llama 3.1 已被微調，使其在調用函數或工具方面更為有效。它還有兩個內建工具，模型可以根據使用者的提示識別出需要使用這些工具。這些工具是： 

- **Brave Search** - 可用於透過網絡搜尋取得如天氣等最新資訊 
- **Wolfram Alpha** - 可用於更複雜的數學計算，無需編寫自己的函數。 

你也可以創建自己的自訂工具，讓 LLM 調用。 

下面的程式碼範例中： 

- 我們在系統提示中定義可用的工具（brave_search, wolfram_alpha）。 
- 發送一個詢問某城市天氣的使用者提示。 
- LLM 會以調用 Brave Search 工具的方式回應，看起來像這樣 `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*注意：這個範例只是進行工具調用，如果你想獲得結果，需在 Brave API 頁面創建一個免費帳號並定義該函數` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

儘管是大型語言模型，Llama 3.1 有一個限制是多模態能力。即能夠使用不同類型的輸入，例如圖片作為提示並提供回應。這項能力是 Llama 3.2 的主要特色之一。這些特色還包括： 

- 多模態 - 能夠同時評估文字和圖片提示 
- 小到中型變體（11B 和 90B） - 提供靈活的部署選項， 
- 僅文字變體（1B 和 3B） - 允許模型部署於邊緣/行動裝置並提供低延遲 

多模態支援是開源模型領域的一大進步。以下程式碼範例接受圖片和文字提示，從 Llama 3.2 90B 獲得圖片分析。 

### Llama 3.2 的多模態支援


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## 學習不會於此停止，繼續您的旅程

完成本課程後，請查看我們的 [生成式 AI 學習合集](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst)，繼續提升您的生成式 AI 知識！


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
